# Markov Decision Processes

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 09.03 |
| Optional | 04.03 |
| Time | ~80 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/09_Advanced_Probability/08_markov_decision_processes.ipynb)

---

## 🎯 Learning Objective

Formalize sequential decision-making under uncertainty as a Markov Decision Process, derive the
Bellman equations, prove convergence of value iteration via the Banach fixed-point theorem, and
connect the framework to modern reinforcement learning and RLHF.

---

## 📐 Theory

`09.03` studied Markov chains — stochastic processes where a system transitions between states
according to fixed probabilities $P_{ij}$, with no external *control*. This notebook extends
that framework by introducing **actions** (the agent *chooses* which transition kernel to
invoke) and **rewards** (the agent wants to maximize long-run payoff), arriving at the
**Markov Decision Process** — the mathematical foundation of all reinforcement learning.

### 1. The MDP framework

A (finite, discounted) **Markov Decision Process** is a tuple $(S, A, P, R, \gamma)$:

- $S = \{1, \ldots, n\}$: a finite **state space** (the set of situations the agent can be in).
- $A = \{1, \ldots, m\}$: a finite **action space** (the set of decisions available at each
  state; more generally $A(s)$ can depend on $s$, but we keep notation simple).
- $P(s' \mid s, a)$: **transition probabilities** — the probability of moving to state $s'$
  given the agent is in state $s$ and takes action $a$. For each fixed $(s,a)$, this is a
  probability distribution over $S$, exactly as in `09.03`'s transition matrix but now
  *indexed by the action taken*. The Markov property is preserved: the distribution of $s'$
  depends only on the current state–action pair, not the history.
- $R(s, a)$: **immediate reward** received when taking action $a$ in state $s$. (Some
  formulations use $R(s, a, s')$; the expectation over $s'$ reduces to the $R(s,a)$ form.)
- $\gamma \in [0, 1)$: **discount factor** — how much the agent values future reward relative
  to present reward. Setting $\gamma$ close to $1$ makes the agent far-sighted; $\gamma$
  close to $0$ makes it myopic.

A **policy** $\pi$ specifies the agent's behavior: $\pi(a \mid s)$ is the probability of
choosing action $a$ in state $s$. A **deterministic policy** $\pi: S \to A$ picks exactly one
action per state. Under a fixed policy, the MDP reduces to a Markov chain on $S$ with
transition matrix $P^\pi_{ss'} = \sum_a \pi(a|s) P(s'|s,a)$ — recovering `09.03`'s setting
exactly.

### 2. Value functions

The **state-value function** under policy $\pi$ measures the expected discounted return
starting from state $s$ and following $\pi$ thereafter:

$$V^\pi(s) = \mathbb{E}_\pi\!\left[\sum_{t=0}^{\infty} \gamma^t R(s_t, a_t) \;\Big|\; s_0 = s\right].$$

The **action-value function** (Q-function) conditions additionally on the first action taken:

$$Q^\pi(s, a) = \mathbb{E}_\pi\!\left[\sum_{t=0}^{\infty} \gamma^t R(s_t, a_t) \;\Big|\; s_0 = s,\, a_0 = a\right].$$

The two are related by marginalizing over the policy's action distribution:

$$V^\pi(s) = \sum_{a} \pi(a|s)\, Q^\pi(s, a),$$

and conversely $Q^\pi(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a)\, V^\pi(s')$. The
geometric sum $\sum_{t=0}^\infty \gamma^t$ converges because $\gamma < 1$, bounding
$|V^\pi(s)| \leq R_{\max} / (1-\gamma)$ for all $s$ and $\pi$.

### 3. Bellman expectation equations

Splitting the return into the immediate reward plus the discounted future gives the
**Bellman expectation equation** for $V^\pi$:

$$V^\pi(s) = \sum_{a} \pi(a|s) \left[R(s,a) + \gamma \sum_{s'} P(s'|s,a)\, V^\pi(s')\right].$$

This is a system of $|S|$ *linear* equations in $|S|$ unknowns (the values $V^\pi(s)$). In
matrix–vector form with $V^\pi \in \mathbb{R}^{|S|}$, the policy-induced reward vector
$r^\pi_s = \sum_a \pi(a|s) R(s,a)$, and the policy-induced transition matrix
$P^\pi_{ss'} = \sum_a \pi(a|s) P(s'|s,a)$:

$$V^\pi = r^\pi + \gamma\, P^\pi V^\pi \quad\Longleftrightarrow\quad (I - \gamma\, P^\pi)\, V^\pi = r^\pi.$$

Since $\gamma < 1$ and $P^\pi$ is row-stochastic (spectral radius $\leq 1$), the matrix
$(I - \gamma P^\pi)$ is non-singular — the linear system has a unique solution, which can be
computed exactly by matrix inversion for small state spaces.

The analogous Bellman equation for $Q^\pi$ is:
$Q^\pi(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) \sum_{a'} \pi(a'|s')\, Q^\pi(s',a')$.

### 4. Bellman optimality equations

The **optimal value function** $V^*(s) = \max_\pi V^\pi(s)$ satisfies the **Bellman optimality
equation**, where the $\max$ replaces the policy-weighted average:

$$V^*(s) = \max_{a \in A} \left[R(s,a) + \gamma \sum_{s'} P(s'|s,a)\, V^*(s')\right].$$

This is a system of $|S|$ *nonlinear* equations (due to the $\max$). A fundamental theorem
of MDPs guarantees that $V^*$ exists, is unique, and that the **greedy policy** with respect
to $V^*$ is optimal:

$$\pi^*(s) = \arg\max_{a \in A} \left[R(s,a) + \gamma \sum_{s'} P(s'|s,a)\, V^*(s')\right].$$

Moreover, there always exists a *deterministic* optimal policy — no randomization is needed.

### 5. Value iteration and the contraction mapping theorem

Define the **Bellman optimality operator** $T: \mathbb{R}^{|S|} \to \mathbb{R}^{|S|}$ by

$$(TV)(s) = \max_{a \in A} \left[R(s,a) + \gamma \sum_{s'} P(s'|s,a)\, V(s')\right].$$

**Value iteration** is the fixed-point iteration $V_{k+1} = TV_k$, starting from any
initial guess $V_0$ (typically all zeros).

**Claim:** $T$ is a $\gamma$-contraction in the $\ell_\infty$ norm, i.e.
$\|TV - TV'\|_\infty \leq \gamma \|V - V'\|_\infty$ for all $V, V'$.

*Proof.* For any state $s$, let $a^*$ achieve the max in $(TV)(s)$. Then

$$\begin{aligned}
(TV)(s) - (TV')(s) &\leq \left[R(s,a^*) + \gamma \sum_{s'} P(s'|s,a^*)V(s')\right]
  - \left[R(s,a^*) + \gamma \sum_{s'} P(s'|s,a^*)V'(s')\right] \\
&= \gamma \sum_{s'} P(s'|s,a^*)\bigl[V(s') - V'(s')\bigr]
  \leq \gamma \|V - V'\|_\infty,
\end{aligned}$$

where the first inequality uses the fact that $a^*$ maximizes the first bracket but
may not maximize the second, and the last inequality uses $\sum_{s'} P(s'|s,a^*) = 1$
and $V(s')-V'(s') \leq \|V-V'\|_\infty$. By symmetry (swapping $V$ and $V'$),
$|(TV)(s)-(TV')(s)| \leq \gamma\|V-V'\|_\infty$ for every $s$, hence
$\|TV-TV'\|_\infty \leq \gamma\|V-V'\|_\infty$. $\square$

By the **Banach fixed-point theorem**, since $(\mathbb{R}^{|S|}, \|\cdot\|_\infty)$ is
a complete metric space and $T$ is a $\gamma$-contraction:
1. $T$ has a **unique** fixed point $V^* = TV^*$ (which is the optimal value function).
2. For any starting $V_0$, $\|V_k - V^*\|_\infty \leq \gamma^k \|V_0 - V^*\|_\infty$ —
   **linear convergence** at rate $\gamma$.

Smaller $\gamma$ means faster convergence (stronger contraction), at the cost of a more
myopic optimal policy.

### 6. Policy iteration

An alternative to value iteration alternates two steps:

1. **Policy evaluation:** given a policy $\pi_k$, solve the linear system
   $(I - \gamma P^{\pi_k}) V^{\pi_k} = r^{\pi_k}$ to find $V^{\pi_k}$ exactly.
2. **Policy improvement:** set $\pi_{k+1}(s) = \arg\max_a [R(s,a) + \gamma \sum_{s'}
   P(s'|s,a) V^{\pi_k}(s')]$ for each $s$.

The **policy improvement theorem** guarantees $V^{\pi_{k+1}}(s) \geq V^{\pi_k}(s)$ for
all $s$, with equality everywhere iff $\pi_k$ is already optimal. Since there are at most
$|A|^{|S|}$ deterministic policies (a finite set), the sequence of strictly improving
policies must terminate — at optimality — in finitely many steps. In practice, policy
iteration typically converges in far fewer iterations than value iteration (often 3–10),
though each iteration is more expensive (solving a linear system vs. a single Bellman
backup).

### 7. Connection to reinforcement learning and Q-learning

The algorithms above require *knowing* $P$ and $R$ — the **planning** setting. When the
agent does *not* know the environment's dynamics (the **learning** setting — reinforcement
learning), it must estimate value functions from experience.

**Q-learning** (Watkins, 1989) estimates $Q^*$ directly from interaction samples
$(s_t, a_t, r_t, s_{t+1})$ via the update:

$$Q(s_t, a_t) \leftarrow Q(s_t, a_t) + \alpha_t \bigl[r_t + \gamma \max_{a'} Q(s_{t+1}, a') - Q(s_t, a_t)\bigr],$$

where $\alpha_t$ is the learning rate. This is a **stochastic approximation** to the
Bellman optimality equation: the term in brackets is a noisy, unbiased sample of the
Bellman residual $[TQ](s,a) - Q(s,a)$, driving $Q$ toward the fixed point $Q^*$.

**Convergence** of Q-learning (Watkins & Dayan, 1992; Tsitsiklis, 1994) follows from
combining two ingredients:
1. The Bellman operator $T$ is a $\gamma$-contraction (proved above) — so the target
   $Q^*$ is a stable fixed point.
2. The stochastic updates form a **supermartingale** (connecting to `09.07`'s theory):
   under the Robbins–Monro conditions on learning rates ($\sum \alpha_t = \infty$,
   $\sum \alpha_t^2 < \infty$) and sufficient exploration, the noise averages out and
   $Q_t \to Q^*$ almost surely.

This contraction + stochastic approximation template underpins virtually all tabular
RL convergence proofs. Deep RL (DQN, PPO, SAC) replaces the table $Q(s,a)$ with a
neural network — gaining scalability but losing the clean convergence guarantees.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.patches import FancyArrowPatch
import matplotlib.cm as cm
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

A 5×5 grid-world MDP with walls, a goal (reward +1), and a penalty cell (reward −1) makes
the Bellman equation's structure immediately visible. The left panel draws the state space
with optimal-policy arrows and the $V^*$ heatmap (states closer to the goal are warmer —
exactly the discounted future reward accumulated by following the optimal path). The right
panel plots $\|V_k - V_{k-1}\|_\infty$ on a log scale across value-iteration sweeps, verifying
the $\gamma^k$ linear convergence rate predicted by the Banach fixed-point theorem.

In [ ]:
# --- 5x5 Grid-World MDP: state space, V* heatmap, value iteration convergence ---
ROWS, COLS = 5, 5
WALLS = {(1, 1), (2, 1), (3, 1), (1, 3)}  # impassable cells
GOAL = (0, 4)       # reward +1, terminal
PENALTY = (4, 4)    # reward -1, terminal
GAMMA = 0.95
STEP_REWARD = -0.04  # small penalty per step encourages reaching goal quickly

ACTIONS = {'up': (-1, 0), 'down': (1, 0), 'left': (0, -1), 'right': (0, 1)}
ACTION_LIST = list(ACTIONS.keys())

def is_valid(r, c):
    return 0 <= r < ROWS and 0 <= c < COLS and (r, c) not in WALLS

def step(s, a):
    """Deterministic transitions for clarity of visualization."""
    dr, dc = ACTIONS[a]
    nr, nc = s[0] + dr, s[1] + dc
    if is_valid(nr, nc):
        return (nr, nc)
    return s  # bounce back if hitting wall or boundary

def reward(s, a):
    ns = step(s, a)
    if ns == GOAL:
        return 1.0
    elif ns == PENALTY:
        return -1.0
    return STEP_REWARD

def is_terminal(s):
    return s == GOAL or s == PENALTY

# Value iteration
V = np.zeros((ROWS, COLS))
errors = []
for iteration in range(200):
    V_new = np.copy(V)
    for r in range(ROWS):
        for c in range(COLS):
            if (r, c) in WALLS or is_terminal((r, c)):
                continue
            q_vals = []
            for a in ACTION_LIST:
                ns = step((r, c), a)
                q_vals.append(reward((r, c), a) + GAMMA * V[ns[0], ns[1]])
            V_new[r, c] = max(q_vals)
    errors.append(np.max(np.abs(V_new - V)))
    V = V_new
    if errors[-1] < 1e-10:
        break

V_star = V.copy()

# Extract optimal policy
policy = {}
for r in range(ROWS):
    for c in range(COLS):
        if (r, c) in WALLS or is_terminal((r, c)):
            continue
        best_a, best_q = None, -np.inf
        for a in ACTION_LIST:
            ns = step((r, c), a)
            q = reward((r, c), a) + GAMMA * V_star[ns[0], ns[1]]
            if q > best_q:
                best_a, best_q = a, q
        policy[(r, c)] = best_a

# --- Visualization ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: V* heatmap + optimal policy arrows
V_display = np.copy(V_star)
for (r, c) in WALLS:
    V_display[r, c] = np.nan

norm = Normalize(vmin=np.nanmin(V_display) - 0.1, vmax=np.nanmax(V_display) + 0.1)
cmap = cm.RdYlGn

for r in range(ROWS):
    for c in range(COLS):
        if (r, c) in WALLS:
            ax1.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, color='#444444'))
            ax1.text(c, r, 'WALL', ha='center', va='center', fontsize=7, color='white', fontweight='bold')
        elif is_terminal((r, c)):
            color = '#2ecc71' if (r, c) == GOAL else '#e74c3c'
            ax1.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, color=color, alpha=0.7))
            label = 'GOAL\n+1' if (r, c) == GOAL else 'PEN\n\u22121'
            ax1.text(c, r, label, ha='center', va='center', fontsize=8, fontweight='bold')
        else:
            color = cmap(norm(V_display[r, c]))
            ax1.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, color=color, alpha=0.6))
            ax1.text(c, r - 0.3, f'{V_star[r,c]:.2f}', ha='center', va='center', fontsize=7)
            # Draw policy arrow
            if (r, c) in policy:
                arrow_map = {'up': (0, -0.25), 'down': (0, 0.25), 'left': (-0.25, 0), 'right': (0.25, 0)}
                dx, dy = arrow_map[policy[(r, c)]]
                ax1.annotate('', xy=(c + dx, r + dy), xytext=(c, r + 0.05),
                            arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.8))

ax1.set_xlim(-0.5, COLS - 0.5)
ax1.set_ylim(ROWS - 0.5, -0.5)
ax1.set_aspect('equal')
ax1.set_xticks(range(COLS))
ax1.set_yticks(range(ROWS))
ax1.set_title('Optimal Value $V^*$ and Policy $\\pi^*$\n(5\u00d75 grid-world, \u03b3 = 0.95)', fontsize=11)
ax1.grid(True, color='gray', linewidth=0.3)

# Right: Convergence plot
errors_arr = np.array(errors)
mask = errors_arr > 1e-15
ax2.semilogy(np.arange(len(errors_arr))[mask], errors_arr[mask], 'o-', color='#4C72B0',
             markersize=3, label='$\\|V_k - V_{k-1}\\|_\\infty$')
# Overlay theoretical gamma^k rate
k_range = np.arange(len(errors))
theoretical = errors[0] * GAMMA**k_range
ax2.semilogy(k_range[theoretical > 1e-15], theoretical[theoretical > 1e-15],
             '--', color='#C44E52', lw=1.5, label=f'$O(\\gamma^k)$, \u03b3={GAMMA}')
ax2.set_xlabel('Iteration k')
ax2.set_ylabel('Bellman residual (log scale)')
ax2.set_title('Value Iteration Convergence\n(linear rate \u03b3 \u2014 Banach fixed-point)', fontsize=11)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()
print(f'Value iteration converged in {len(errors)} iterations (residual < 1e-10).')
print(f'Discount factor \u03b3 = {GAMMA} \u2192 contraction rate = {GAMMA}, '
      f'theoretical iterations to 1e-10: {int(np.ceil(np.log(1e-10 / errors[0]) / np.log(GAMMA)))}')

## 🐍 Implementation from Scratch

We implement the full MDP toolchain on a 4×4 grid-world with stochastic transitions (the
intended action succeeds with probability 0.8; with probability 0.1 each, the agent slips
sideways). Four independent implementations — value iteration, policy iteration, direct
Bellman-equation solve, and tabular Q-learning — are cross-checked against each other,
confirming that all four converge to the same $V^*$ and $\pi^*$.

In [ ]:
# --- 4x4 Grid-World MDP with stochastic transitions ---
# Value Iteration, Policy Iteration, and Tabular Q-Learning from scratch.

ROWS, COLS = 4, 4
WALLS = {(1, 1)}
GOAL = (0, 3)        # +1, terminal
PENALTY = (1, 3)     # -1, terminal
GAMMA = 0.9
STEP_REWARD = -0.04
TERMINALS = {GOAL, PENALTY}

ACTIONS = {'up': (-1, 0), 'down': (1, 0), 'left': (0, -1), 'right': (0, 1)}
ACTION_LIST = list(ACTIONS.keys())
ACT_IDX = {a: i for i, a in enumerate(ACTION_LIST)}
PERPENDICULAR = {
    'up': ['left', 'right'], 'down': ['left', 'right'],
    'left': ['up', 'down'], 'right': ['up', 'down']
}

# All non-wall states
states = [(r, c) for r in range(ROWS) for c in range(COLS) if (r, c) not in WALLS]
state_idx = {s: i for i, s in enumerate(states)}
n_states = len(states)
n_actions = len(ACTION_LIST)

def move(s, action_name):
    dr, dc = ACTIONS[action_name]
    nr, nc = s[0] + dr, s[1] + dc
    if 0 <= nr < ROWS and 0 <= nc < COLS and (nr, nc) not in WALLS:
        return (nr, nc)
    return s

# Build transition probability table P[s_idx, a_idx, s'_idx] and reward R[s_idx, a_idx]
P = np.zeros((n_states, n_actions, n_states))
R = np.zeros((n_states, n_actions))

for s in states:
    si = state_idx[s]
    if s in TERMINALS:
        # Terminal: self-loop with zero reward (episode ends)
        for ai in range(n_actions):
            P[si, ai, si] = 1.0
            R[si, ai] = 0.0
        continue
    for a_name in ACTION_LIST:
        ai = ACT_IDX[a_name]
        # Stochastic: 0.8 intended, 0.1 each perpendicular
        transitions = [(move(s, a_name), 0.8)]
        for perp in PERPENDICULAR[a_name]:
            transitions.append((move(s, perp), 0.1))
        # Compute expected reward and transition probs
        r_total = 0.0
        for (ns, prob) in transitions:
            nsi = state_idx[ns]
            P[si, ai, nsi] += prob
            if ns == GOAL:
                r_total += prob * 1.0
            elif ns == PENALTY:
                r_total += prob * (-1.0)
            else:
                r_total += prob * STEP_REWARD
        R[si, ai] = r_total

# Sanity: transition probabilities sum to 1
assert np.allclose(P.sum(axis=2), 1.0), "Transition probs must sum to 1"

# ============================
# 1. VALUE ITERATION
# ============================
V_vi = np.zeros(n_states)
vi_errors = []
for k in range(500):
    Q_all = R + GAMMA * (P @ V_vi)  # shape (n_states, n_actions)
    V_new = Q_all.max(axis=1)
    vi_errors.append(np.max(np.abs(V_new - V_vi)))
    V_vi = V_new
    if vi_errors[-1] < 1e-10:
        break

Q_star_vi = R + GAMMA * (P @ V_vi)
policy_vi = Q_star_vi.argmax(axis=1)
print(f"Value Iteration converged in {len(vi_errors)} iterations")

# ============================
# 2. POLICY ITERATION
# ============================
policy_pi = np.zeros(n_states, dtype=int)  # start with action 0 everywhere
pi_iters = 0
for _ in range(100):
    # Policy evaluation: solve (I - gamma * P^pi) V = r^pi
    P_pi = np.array([P[s, policy_pi[s], :] for s in range(n_states)])  # (n_states, n_states)
    r_pi = np.array([R[s, policy_pi[s]] for s in range(n_states)])
    V_pi = np.linalg.solve(np.eye(n_states) - GAMMA * P_pi, r_pi)

    # Policy improvement
    Q_all = R + GAMMA * (P @ V_pi)
    new_policy = Q_all.argmax(axis=1)
    pi_iters += 1
    if np.array_equal(new_policy, policy_pi):
        break
    policy_pi = new_policy

V_pi_final = V_pi
print(f"Policy Iteration converged in {pi_iters} iterations")

# ============================
# 3. TABULAR Q-LEARNING
# ============================
rng = np.random.default_rng(42)
Q_ql = np.zeros((n_states, n_actions))
alpha_0 = 0.5
epsilon = 0.2
n_episodes = 20_000
visit_count = np.zeros((n_states, n_actions))

non_terminal_states = [s for s in states if s not in TERMINALS]

for ep in range(n_episodes):
    s = non_terminal_states[rng.integers(len(non_terminal_states))]
    si = state_idx[s]
    for t in range(100):  # max steps per episode
        # Epsilon-greedy action selection
        if rng.random() < epsilon:
            ai = rng.integers(n_actions)
        else:
            ai = Q_ql[si].argmax()

        # Sample next state from P[si, ai, :]
        nsi = rng.choice(n_states, p=P[si, ai])
        r = R[si, ai]

        # Q-learning update with decaying learning rate
        visit_count[si, ai] += 1
        alpha = alpha_0 / (1 + 0.001 * visit_count[si, ai])
        td_target = r + GAMMA * Q_ql[nsi].max()
        Q_ql[si, ai] += alpha * (td_target - Q_ql[si, ai])

        si = nsi
        if states[si] in TERMINALS:
            break

V_ql = Q_ql.max(axis=1)
policy_ql = Q_ql.argmax(axis=1)

# ============================
# CROSS-CHECK: all three methods
# ============================
print(f"\n{'State':<10} {'V* (VI)':>10} {'V* (PI)':>10} {'V* (QL)':>10} {'\u03c0*(VI)':>8} {'\u03c0*(PI)':>8} {'\u03c0*(QL)':>8}")
print('-' * 70)
for s in states:
    si = state_idx[s]
    print(f"{str(s):<10} {V_vi[si]:10.4f} {V_pi_final[si]:10.4f} {V_ql[si]:10.4f} "
          f"{ACTION_LIST[policy_vi[si]]:>8} {ACTION_LIST[policy_pi[si]]:>8} {ACTION_LIST[policy_ql[si]]:>8}")

print(f"\nV* agreement (VI vs PI): max|diff| = {np.max(np.abs(V_vi - V_pi_final)):.2e}")
print(f"V* agreement (VI vs QL): max|diff| = {np.max(np.abs(V_vi - V_ql)):.4f}  (QL is stochastic, so approximate)")
print(f"Policy agreement (VI vs PI): {np.array_equal(policy_vi, policy_pi)}")
# Q-learning policy may differ at states where multiple actions are near-optimal
policy_match = np.mean(policy_vi == policy_ql)
print(f"Policy agreement (VI vs QL): {policy_match:.0%} of states")

## 🤖 Why This Matters for AI

The MDP framework is not a historical curiosity — it is the *active mathematical language*
of the most consequential AI systems deployed today:

- **RLHF** (Reinforcement Learning from Human Feedback) — the training paradigm behind
  ChatGPT, Claude, and Gemini — is an MDP in disguise. The **state** is the token sequence
  generated so far, the **action** is the next token (from the vocabulary), the **transition**
  is deterministic (append the token), and the **reward** is provided at the end by a learned
  reward model trained on human preference comparisons. The policy (the language model itself)
  is optimized via PPO — a policy-gradient algorithm whose convergence analysis rests on the
  same Bellman-equation structure derived above. The KL penalty against the reference model
  acts as a modified discount/regularization term in the MDP objective.

- **AlphaGo / AlphaZero** formulates the game of Go as an MDP (state = board position,
  action = stone placement) and uses Monte Carlo Tree Search (MCTS) to approximate the
  Bellman optimality equation via rollout simulations — a model-based RL approach.

- **Multi-step reasoning** in LLMs (chain-of-thought, tree-of-thought, beam search) can be
  formalized as planning in an MDP where each "reasoning state" is a partial chain of
  thought, each "action" extends it by one step, and the reward reflects answer quality.
  This MDP perspective suggests that better *search algorithms* (not just bigger models)
  may unlock stronger reasoning.

- The **Bellman equation** is, as Sutton & Barto put it, the "single most important equation
  in reinforcement learning." Every RL algorithm — from tabular Q-learning to PPO to
  offline RL — either solves, approximates, or samples from some variant of the Bellman
  equation. Understanding its mathematical structure (contraction, fixed point, linear system
  for evaluation, nonlinear system for optimality) is prerequisite to reading any modern RL
  or RLHF paper.

In [ ]:
# --- Toy Token MDP: RLHF as an MDP + Q-learning vs Value Iteration comparison ---
#
# A simplified language-generation MDP: the agent builds a 3-token response from a
# small vocabulary. The reward model scores the complete sequence at the end.
# We solve with value iteration (known dynamics) and Q-learning (learned from samples),
# then compare convergence.

VOCAB = ['good', 'great', 'bad', 'ok']
MAX_LEN = 3
GAMMA_TOK = 0.99  # near 1: reward at end, no heavy discounting

# Build all possible states: tuples of tokens generated so far (including empty start)
from itertools import product

all_states = [()]  # start state
for length in range(1, MAX_LEN + 1):
    for seq in product(range(len(VOCAB)), repeat=length):
        all_states.append(seq)

state_to_idx = {s: i for i, s in enumerate(all_states)}
n_tok_states = len(all_states)
n_vocab = len(VOCAB)

def is_tok_terminal(s):
    return len(s) == MAX_LEN

def reward_model(seq):
    """Toy reward model: prefers sequences with 'good'/'great', penalizes 'bad'."""
    score = 0.0
    for tok_id in seq:
        if VOCAB[tok_id] == 'great':
            score += 1.0
        elif VOCAB[tok_id] == 'good':
            score += 0.5
        elif VOCAB[tok_id] == 'bad':
            score -= 1.0
        else:  # 'ok'
            score += 0.1
    return score

# Build transition and reward tables
P_tok = np.zeros((n_tok_states, n_vocab, n_tok_states))
R_tok = np.zeros((n_tok_states, n_vocab))

for s in all_states:
    si = state_to_idx[s]
    if is_tok_terminal(s):
        for ai in range(n_vocab):
            P_tok[si, ai, si] = 1.0  # absorbing
            R_tok[si, ai] = 0.0
    else:
        for ai in range(n_vocab):
            ns = s + (ai,)
            nsi = state_to_idx[ns]
            P_tok[si, ai, nsi] = 1.0
            # Reward only at terminal states
            if is_tok_terminal(ns):
                R_tok[si, ai] = reward_model(ns)
            else:
                R_tok[si, ai] = 0.0

# Value iteration on token MDP
V_tok = np.zeros(n_tok_states)
vi_tok_errors = []
for k in range(100):
    Q_tok = R_tok + GAMMA_TOK * (P_tok @ V_tok)
    V_new_tok = Q_tok.max(axis=1)
    vi_tok_errors.append(np.max(np.abs(V_new_tok - V_tok)))
    V_tok = V_new_tok
    if vi_tok_errors[-1] < 1e-12:
        break

# Decode optimal policy from start state
s = ()
optimal_sequence = []
for t in range(MAX_LEN):
    si = state_to_idx[s]
    Q_s = R_tok[si] + GAMMA_TOK * (P_tok[si] @ V_tok)
    best_a = Q_s.argmax()
    optimal_sequence.append(VOCAB[best_a])
    s = s + (best_a,)

print("=== Toy RLHF Token MDP ===")
print(f"Vocabulary: {VOCAB}")
print(f"Sequence length: {MAX_LEN}")
print(f"Total states: {n_tok_states}")
print(f"Optimal response (value iteration): {' \u2192 '.join(optimal_sequence)}")
print(f"Optimal return: {V_tok[state_to_idx[()]]:.4f}")
print(f"Reward of optimal sequence: {reward_model(tuple(VOCAB.index(w) for w in optimal_sequence)):.2f}")

# Q-learning on the same token MDP
rng = np.random.default_rng(42)
Q_ql_tok = np.zeros((n_tok_states, n_vocab))
ql_value_errors = []  # track |V_ql - V*| over episodes
alpha_0_tok = 0.3
eps_tok = 0.3
n_ql_episodes = 5000
visit_tok = np.zeros((n_tok_states, n_vocab))

for ep in range(n_ql_episodes):
    s = ()
    si = state_to_idx[s]
    for t in range(MAX_LEN):
        if rng.random() < eps_tok:
            ai = rng.integers(n_vocab)
        else:
            ai = Q_ql_tok[si].argmax()
        ns = s + (ai,)
        nsi = state_to_idx[ns]
        r = R_tok[si, ai]
        visit_tok[si, ai] += 1
        alpha = alpha_0_tok / (1 + 0.002 * visit_tok[si, ai])
        Q_ql_tok[si, ai] += alpha * (r + GAMMA_TOK * Q_ql_tok[nsi].max() - Q_ql_tok[si, ai])
        s, si = ns, nsi
        if is_tok_terminal(s):
            break
    if ep % 50 == 0:
        V_ql_tok = Q_ql_tok.max(axis=1)
        ql_value_errors.append(np.max(np.abs(V_ql_tok - V_tok)))

# --- Convergence comparison plot ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Value iteration convergence (token MDP)
ax1.semilogy(vi_tok_errors, 'o-', color='#4C72B0', markersize=3)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Bellman residual (log scale)')
ax1.set_title('Value Iteration on Token MDP\n(converges in a few sweeps \u2014 known dynamics)', fontsize=10)

# Right: Q-learning convergence
ax2.semilogy(np.arange(0, n_ql_episodes, 50), ql_value_errors, 's-', color='#C44E52', markersize=3)
ax2.set_xlabel('Episode')
ax2.set_ylabel('$\\|V_{QL} - V^*\\|_\\infty$ (log scale)')
ax2.set_title('Q-Learning on Token MDP\n(slower \u2014 must learn dynamics from samples)', fontsize=10)

plt.tight_layout()
plt.show()

# Decode Q-learning's policy
s = ()
ql_sequence = []
for t in range(MAX_LEN):
    si = state_to_idx[s]
    best_a = Q_ql_tok[si].argmax()
    ql_sequence.append(VOCAB[best_a])
    s = s + (best_a,)
print(f"\nQ-learning's best response: {' \u2192 '.join(ql_sequence)}")
print(f"Match with VI optimal: {ql_sequence == optimal_sequence}")
print("\nKey insight: VI needs the transition model P (planning); Q-learning learns from")
print("samples alone (RL). Both converge to the same Q* \u2014 the Bellman fixed point.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For a 3-state MDP with states $\{1,2,3\}$, single action per state (so it's just a
   Markov reward process), transition matrix
   $P = \begin{pmatrix}0 & 0.5 & 0.5\\\ 0 & 0 & 1\\\ 1 & 0 & 0\end{pmatrix}$,
   rewards $R = (1, 2, 0)^\top$, and $\gamma = 0.9$, compute $V^\pi$ by solving the
   linear system $(I - \gamma P) V = R$ by hand.

<details>
<summary>💡 Solution</summary>

$(I - 0.9P)V = R$ gives:

$V_1 - 0.45V_2 - 0.45V_3 = 1$

$V_2 - 0.9V_3 = 2$

$-0.9V_1 + V_3 = 0$

From equation 3: $V_3 = 0.9V_1$. Substituting into equation 2: $V_2 = 2 + 0.81V_1$.
Substituting both into equation 1:
$V_1 - 0.45(2 + 0.81V_1) - 0.45(0.9V_1) = 1$,
so $V_1(1 - 0.3645 - 0.405) = 1.9$, giving $V_1 = 1.9/0.2305 \approx 8.243$.
Then $V_3 \approx 7.419$, $V_2 \approx 8.677$.
Verify: `np.linalg.solve(np.eye(3) - 0.9*P, R)` returns $[8.243, 8.677, 7.419]$.

</details>

### 💭 UNDERSTAND
2. Why does value iteration converge *faster* when $\gamma$ is small? Your answer should
   reference the contraction constant explicitly and explain the intuitive trade-off.

<details>
<summary>💡 Solution</summary>

The Bellman operator $T$ is a $\gamma$-contraction: $\|TV - TV'\|_\infty \leq \gamma
\|V - V'\|_\infty$. After $k$ iterations, $\|V_k - V^*\|_\infty \leq \gamma^k
\|V_0 - V^*\|_\infty$. A smaller $\gamma$ means a stronger contraction (faster geometric
decay), so fewer iterations to reach a given precision. The trade-off: small $\gamma$ makes
the agent near-sighted (it mostly cares about immediate rewards), so the resulting $V^*$ may
represent a worse long-horizon strategy. Large $\gamma$ (close to 1) captures long-run
consequences but the contraction is weak, requiring many more iterations.

</details>

### ✏️ DERIVE
3. Prove that the Bellman optimality operator $T$ is a $\gamma$-contraction in the
   $\ell_\infty$ norm. (Hint: the key step is using $\max_a f(a) - \max_a g(a) \leq
   \max_a [f(a) - g(a)]$ and the fact that $P(\cdot|s,a)$ sums to 1.)

<details>
<summary>💡 Solution outline</summary>

For any state $s$:

$(TV)(s) - (TV')(s) = \max_a[R(s,a) + \gamma \sum_{s'}P(s'|s,a)V(s')] - \max_a[R(s,a) + \gamma \sum_{s'}P(s'|s,a)V'(s')]$

Using $\max f - \max g \leq \max(f-g)$: let $a^*$ achieve the max in $(TV)(s)$. Then
$(TV)(s) - (TV')(s) \leq R(s,a^*) + \gamma\sum_{s'}P(s'|s,a^*)V(s') - R(s,a^*) - \gamma\sum_{s'}P(s'|s,a^*)V'(s')$
$= \gamma\sum_{s'}P(s'|s,a^*)[V(s')-V'(s')] \leq \gamma\|V-V'\|_\infty$ since $\sum_{s'}P(s'|s,a^*)=1$.
By symmetry (swap $V,V'$), $|(TV)(s)-(TV')(s)| \leq \gamma\|V-V'\|_\infty$ for all $s$,
so $\|TV-TV'\|_\infty \leq \gamma\|V-V'\|_\infty$. $\square$

</details>

### 🐍 IMPLEMENT
4. Implement Q-learning with $\varepsilon$-greedy exploration ($\varepsilon = 0.1$) on the
   4×4 grid-world defined in the Implementation cell. Run 50,000 episodes and verify
   convergence to the same $Q^*$ as value iteration (max absolute difference < 0.05).

<details>
<summary>✅ How to know you're right</summary>

After 50,000 episodes with a decaying learning rate $\alpha_t = 0.5/(1 + 0.001 \cdot
\text{visits}(s,a))$ and $\varepsilon = 0.1$, the max $|Q_{\text{QL}} - Q^*_{\text{VI}}|$
should be well below 0.05 for all state–action pairs that are visited frequently. States
near terminals converge fastest (shorter episodes, more updates); distant states with
rare visits may have slightly larger residual error — this is expected and illustrates the
exploration–exploitation trade-off.

</details>

### 🤖 APPLY
5. Model a simplified RLHF problem as an MDP: the state is a partial response (up to token
   $t$), the action is the next token from a vocabulary of 5 words, and the reward is given
   only at the end (a "human preference score" based on word content). Run value iteration
   to find the optimal response of length 4. Then add a KL penalty term
   $-\beta \log(\pi(a|s)/\pi_{\text{ref}}(a|s))$ to the reward (where $\pi_{\text{ref}}$
   is a uniform reference policy) and observe how $\beta$ changes the optimal response.

<details>
<summary>✅ What you should observe</summary>

Without KL penalty ($\beta=0$), the optimal policy always picks the highest-reward word.
With increasing $\beta$, the KL penalty discourages deviating from the uniform reference
policy, so the optimal policy becomes more "diverse" — choosing a mix of words rather than
always the best one. At very large $\beta$, the policy approximately matches the reference
(uniform). This exactly mirrors real RLHF: the KL penalty prevents the language model from
"reward hacking" by producing degenerate, repetitive outputs that exploit the reward model.

</details>

### 🚀 CHALLENGE
6. **LP formulation of MDPs.** Show that finding $V^*$ is equivalent to the linear program
   $\min_V \sum_s V(s)$ subject to $V(s) \geq R(s,a) + \gamma \sum_{s'} P(s'|s,a) V(s')$
   for all $s, a$. Derive this from the Bellman optimality equation. Then solve the 4×4
   grid-world MDP using `scipy.optimize.linprog` and verify the solution matches value
   iteration's $V^*$.

<details>
<summary>🔍 What a strong answer covers</summary>

The Bellman optimality equation says $V^*(s) = \max_a[R(s,a) + \gamma\sum_{s'}P(s'|s,a)V^*(s')]$,
which means $V^*(s) \geq R(s,a) + \gamma\sum_{s'}P(s'|s,a)V^*(s')$ for *all* $a$, with equality
for the optimal action. Any $V$ satisfying all these inequalities has $V(s) \geq V^*(s)$ for all
$s$ (provable by induction on the Bellman operator). Therefore $V^*$ is the componentwise
smallest feasible solution, and minimizing $\sum_s V(s)$ (or any positive-weight sum) over the
feasible set yields $V^*$. Implementation: flatten the $|S| \times |A|$ constraints into
`scipy.optimize.linprog`'s $A_{ub} x \leq b_{ub}$ form (negate the constraints to get $\leq$).
The solution should match VI's $V^*$ to within numerical tolerance ($\sim 10^{-8}$).

</details>

---

## 📚 Further Reading
- Sutton & Barto, *Reinforcement Learning: An Introduction* (2018), Ch. 3–4 — the standard
  pedagogical reference for MDPs, Bellman equations, and dynamic programming
- Puterman, *Markov Decision Processes* (2005) — the rigorous mathematical treatment
- Bertsekas & Tsitsiklis, *Neuro-Dynamic Programming* (1996) — convergence theory for
  stochastic approximation and Q-learning
- Ouyang et al., ["Training Language Models to Follow Instructions with Human Feedback"](https://arxiv.org/abs/2203.02155) (2022) — InstructGPT / RLHF

---

*End of Module 09*